# 🛡️ CyberRott 1.0 - Treinamento do Modelo Especialista em Cibersegurança

Este notebook executa o fine-tuning do modelo **Microsoft Phi-4-mini (3.8B)** no Google Colab Pro com **GPU L4 (24GB)** ou **H100 (80GB)**.

### Pipeline de Fusão e Raciocínio:
1. **Verificação da GPU** (BF16 & Flash Attention 2)
2. **Instalação das dependências**
3. **Montagem do Google Drive** (checkpoints e persistência)
4. **Fusão de Dados (Anchor Mixing + Upsampling + GooseReason Streaming + Shuffling)**
5. **Treinamento LoRA / SFT com TRL e ChatML do Phi-4**
6. **Inferência de Teste** (Cadeia de Raciocínio com `<think>...</think>`)

In [ ]:
# 1. Verificar GPU alocada no Colab
!nvidia-smi


In [ ]:
# 2. Instalar dependências modernas para treinamento e aceleração (versões pinadas e compatíveis)
# NOTA: Caso o Colab apresente avisos de bibliotecas, reinicie a sessão do kernel após a instalação.
!pip install -q "transformers>=4.46.0,<5.0.0" "peft>=0.13.0,<0.16.0" "trl>=0.12.0,<0.17.0" "accelerate>=1.0.0" "datasets>=3.0.0" "bitsandbytes>=0.44.0"
# Se estiver em GPU Ampere/Ada/Hopper (L4 ou H100), instalamos flash-attn:
!pip install -q flash-attn --no-build-isolation || echo "Flash Attention não compilado, usando SDPA nativo do PyTorch."


In [ ]:
# 3. Montar Google Drive para persistência dos checkpoints (opcional, recomendado no Colab)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 4. Pipeline de Fusão de Dados (Anchor Mixing + GooseReason + Shuffling)
import os
import re
import json
import random
import shutil
from typing import List, Dict, Any

FINAL_TRAIN_FILE = "/content/cyberrott_final_train.jsonl" if os.path.exists("/content") else "cyberrott_final_train.jsonl"

# 4.0 Verificação de Cache / Pré-existência do Dataset Final
cached_candidates = [
    "/content/cyberrott_final_train.jsonl",
    "/content/drive/MyDrive/cyberrott_final_train.jsonl",
    "/content/drive/MyDrive/CyberRott/cyberrott_final_train.jsonl",
    "cyberrott_final_train.jsonl"
]

existing_dataset = None
for candidate in cached_candidates:
    if os.path.exists(candidate) and os.path.getsize(candidate) > 0:
        existing_dataset = candidate
        break

if existing_dataset:
    if os.path.abspath(existing_dataset) != os.path.abspath(FINAL_TRAIN_FILE):
        print(f"[✓] Dataset final já existe em: {existing_dataset}")
        print(f"[*] Copiando para {FINAL_TRAIN_FILE}...")
        shutil.copy(existing_dataset, FINAL_TRAIN_FILE)
    else:
        print(f"[✓] Dataset final pronto já disponível em: {FINAL_TRAIN_FILE}")

    if os.path.abspath("cyberrott_final_train.jsonl") != os.path.abspath(FINAL_TRAIN_FILE):
        try:
            shutil.copy(FINAL_TRAIN_FILE, "cyberrott_final_train.jsonl")
        except Exception:
            pass

    with open(FINAL_TRAIN_FILE, "r", encoding="utf-8") as f:
        sample_count = sum(1 for line in f if line.strip())

    print("=" * 70)
    print(f"🎉 DATASET JÁ ESTÁ PRONTO (CARREGAMENTO INSTANTÂNEO EM 1 SEGUNDO!): {FINAL_TRAIN_FILE}")
    print(f"• Total de amostras prontas para o SFT: {sample_count}")
    print("• Todo o processo de streaming e filtragem foi pulado com sucesso!")
    print("=" * 70)
else:
    print("[*] Dataset final não encontrado em cache. Iniciando pipeline de geração...")
    ANCHOR_REPEAT = 20
    MAX_HF_SAMPLES = 2500
    HF_DATASET_NAME = "nvidia/Nemotron-Research-GooseReason-0.7M"
    OUTPUT_TRAIN_FILE = FINAL_TRAIN_FILE
    SEED = 42

    CYBER_KEYWORDS = [
        "vulnerability", "exploit", "cve", "cve-", "buffer overflow", "stack overflow",
        "heap overflow", "use-after-free", "out-of-bounds", "format string",
        "race condition", "toctou", "pwn", "shellcode", "rop chain", "ret2libc",
        "privilege escalation", "bypass", "taint analysis", "injection",
        "sql injection", "sqli", "cross-site scripting", "xss", "csrf", "ssrf",
        "command injection", "path traversal", "directory traversal",
        "reverse engineering", "disassembly", "decompiler", "decompilation",
        "gdb", "radare2", "ghidra", "ida pro", "binwalk", "objdump",
        "elf", "pe binary", "assembly", "x86", "x86_64", "arm64",
        "malware", "ransomware", "backdoor", "trojan", "rootkit", "keylogger",
        "botnet", "forensics", "memory dump", "volatility", "sysmon",
        "wireshark", "pcap", "packet analysis", "network traffic", "yara", "sigma",
        "tcp/ip", "packet", "dns", "dns tunneling", "socket", "http request",
        "firewall", "ids", "ips", "edr", "mitre", "port scan", "nmap",
        "cryptography", "rsa", "aes", "elliptic curve", "diffie-hellman",
        "sha256", "jwt", "hash collision", "nonce reuse", "kerberos",
        "active directory", "golden ticket", "silver ticket", "pass-the-hash",
        "authentication", "authorization", "sandbox", "kernel exploit", "container escape"
    ]

    patterns = [rf"\b{re.escape(kw)}\b" if len(kw) <= 4 else re.escape(kw) for kw in CYBER_KEYWORDS]
    cyber_regex = re.compile("|".join(patterns), re.IGNORECASE)

    # 4.1 Carregar Amostras Âncora de Ouro
    anchor_candidates = [
        "dataset_cyber_ready.jsonl",
        "/content/dataset_cyber_ready.jsonl",
        "/content/drive/MyDrive/dataset_cyber_ready.jsonl",
        "/content/drive/MyDrive/CyberRott/dataset_cyber_ready.jsonl"
    ]
    anchor_path = next((p for p in anchor_candidates if os.path.exists(p)), None)
    raw_anchors = []

    if anchor_path:
        print(f"[✓] Carregando amostras âncora a partir de: {anchor_path}")
        with open(anchor_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        raw_anchors.append(json.loads(line))
                    except Exception:
                        pass
    else:
        if os.path.exists("generate_complex_samples.py") and os.path.exists("prepare_dataset.py"):
            print("[*] Gerando amostras âncora via generate_complex_samples.py...")
            os.system("python generate_complex_samples.py && python prepare_dataset.py")
            if os.path.exists("dataset_cyber_ready.jsonl"):
                with open("dataset_cyber_ready.jsonl", "r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            raw_anchors.append(json.loads(line.strip()))

    if not raw_anchors:
        print("[!] Amostras âncora externas não encontradas. Utilizando exemplo de ouro embutido...")
        raw_anchors = [
            {
                "messages": [
                    {
                        "role": "system",
                        "content": "Você é o CyberRott, uma IA especialista em segurança ofensiva, defensiva e análise forense com raciocínio analítico profundo."
                    },
                    {
                        "role": "user",
                        "content": "Realize uma auditoria de segurança no código C abaixo:\n\n```c\nvoid copy_data(char *src) {\n    char buf[128];\n    strcpy(buf, src);\n}\n```"
                    },
                    {
                        "role": "assistant",
                        "content": "<think>\n**Análise de Superfície**: A função `copy_data` copia dados de `src` para `buf[128]` na stack.\n**Identificação de Pontos de Entrada / Vetores**: `src` é fornecido externamente sem validação de tamanho.\n**Mapeamento de Fluxo / Mecânica Interna**: A função `strcpy` não checa o comprimento antes de escrever em `buf`.\n**Detecção de Sinks Perigosos / Falhas de Controle**: Uso inseguro de `strcpy`.\n**Avaliação de Sanitização e Mitigações**: Sanitização ausente.\n**Conclusão da Vulnerabilidade / Anomalia**: Stack-based Buffer Overflow (CWE-121).\n**Impacto Potencial**: Controle do ponteiro de instrução (RIP) ou DoS.\n</think>\n\nFATO: O código é vulnerável a Stack-Based Buffer Overflow (CWE-121) pelo uso inseguro de `strcpy` em buffer de 128 bytes.\n\nSUGESTÃO DE CORREÇÃO:\n```c\n#include <string.h>\nvoid copy_data_fixed(const char *src) {\n    char buf[128];\n    if (!src) return;\n    strncpy(buf, src, sizeof(buf) - 1);\n    buf[sizeof(buf) - 1] = '\\0';\n}\n```"
                    }
                ]
            }
        ]

    # 4.2 Upsampling das Âncoras de Ouro
    anchor_samples = raw_anchors * ANCHOR_REPEAT
    print(f"[✓] Âncoras ponderadas: {len(raw_anchors)} únicas -> {len(anchor_samples)} no treino ({ANCHOR_REPEAT}x upsampling).")

    # 4.3 Streaming e Filtragem do GooseReason-0.7M (NVIDIA)
    hf_cyber_samples = []
    try:
        from datasets import load_dataset
        print(f"[*] Conectando ao streaming de '{HF_DATASET_NAME}'...")
        ds = None
        for split_name in ["train", "code", "stem", "math"]:
            try:
                ds = load_dataset(HF_DATASET_NAME, split=split_name, streaming=True)
                print(f"[✓] Conectado via split='{split_name}'.")
                break
            except Exception:
                continue

        if ds is not None and MAX_HF_SAMPLES > 0:
            for item in ds:
                text_to_check = " ".join(str(v) for v in item.values() if isinstance(v, (str, list, int, float)))
                if cyber_regex.search(text_to_check):
                    question = item.get("problem") or item.get("question") or item.get("prompt") or item.get("instruction") or item.get("text")
                    options = item.get("options")
                    answer_raw = item.get("answer")
                    solution = None

                    if options and isinstance(options, list) and answer_raw:
                        ans_str = str(answer_raw).strip().upper()
                        if len(ans_str) == 1 and "A" <= ans_str <= "Z":
                            idx = ord(ans_str) - ord("A")
                            if 0 <= idx < len(options):
                                solution = f"Opção correta: **{ans_str}**\n\n```\n{options[idx]}\n```"
                            else:
                                solution = f"Opção correta: \\boxed{{{ans_str}}}"

                    if not solution:
                        solution = item.get("solution") or item.get("answer") or item.get("response") or item.get("output")

                    if question and solution:
                        hf_cyber_samples.append({
                            "messages": [
                                {"role": "system", "content": "Você é o CyberRott, uma IA especialista em segurança ofensiva, defensiva e análise forense com raciocínio analítico profundo."}, 
                                {"role": "user", "content": str(question).strip()},
                                {"role": "assistant", "content": str(solution).strip()}
                            ]
                        })
                        if len(hf_cyber_samples) % 100 == 0:
                            print(f"[*] GooseReason-Cyber filtrado: {len(hf_cyber_samples)}/{MAX_HF_SAMPLES} amostras...")
                        if len(hf_cyber_samples) >= MAX_HF_SAMPLES:
                            break
            print(f"[✓] Amostras GooseReason-Cyber filtradas: {len(hf_cyber_samples)}")
    except Exception as e:
        print(f"[Aviso] Streaming Hugging Face indisponível ({e}). Continuando com as amostras âncora.")

    # 4.4 Fusão Final (Anchor Mixing + Shuffling)
    final_dataset = anchor_samples + hf_cyber_samples
    random.seed(SEED)
    random.shuffle(final_dataset)

    with open(OUTPUT_TRAIN_FILE, "w", encoding="utf-8") as f_out:
        for row in final_dataset:
            f_out.write(json.dumps(row, ensure_ascii=False) + "\n")

    if os.path.abspath("cyberrott_final_train.jsonl") != os.path.abspath(OUTPUT_TRAIN_FILE):
        try:
            shutil.copy(OUTPUT_TRAIN_FILE, "cyberrott_final_train.jsonl")
        except Exception:
            pass

    # 4.5 Backup AUTOMÁTICO para o Google Drive
    if os.path.exists("/content/drive/MyDrive"):
        try:
            drive_backup_file = "/content/drive/MyDrive/cyberrott_final_train.jsonl"
            shutil.copy(OUTPUT_TRAIN_FILE, drive_backup_file)
            print(f"[✓] Backup AUTOMÁTICO concluído no Google Drive: {drive_backup_file}")
        except Exception as e:
            print(f"[Aviso] Falha ao realizar backup no Google Drive: {e}")

    print("=" * 70)
    print(f"🎉 DATASET PRONTO PARA TREINAMENTO: {OUTPUT_TRAIN_FILE}")
    print(f"• Âncoras de Ouro (com upsampling {ANCHOR_REPEAT}x): {len(anchor_samples)}")
    print(f"• GooseReason-Cyber Filtrado:               {len(hf_cyber_samples)}")
    print(f"• TOTAL COMBINADO PARA O SFT:               {len(final_dataset)}")
    print("=" * 70)

In [ ]:
# 5. Configurar e Iniciar o Treinamento LoRA (SFTTrainer)
import os
import gc
import inspect
import torch

# Otimizações de alocação de memória CUDA e desativação do W&B
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

# Limpeza prévia de VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_ID = "microsoft/Phi-4-mini-instruct"
OUTPUT_DIR = "/content/drive/MyDrive/CyberRott-Phi4-LoRA" # ou "./CyberRott-LoRA"

print("1. Carregando Tokenizer...")
# trust_remote_code=False: Phi-4 possui suporte nativo no transformers recente
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("2. Carregando Modelo Base em bfloat16...")
attn_impl = "flash_attention_2" if torch.cuda.is_bf16_supported() else "sdpa"
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation=attn_impl,
        trust_remote_code=False
    )
except Exception as e:
    print(f"[Aviso] Falha com {attn_impl}: {e}. Alternando para 'sdpa'...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
        trust_remote_code=False
    )

# 3. Configuração LoRA adaptada para a arquitetura fundida do Phi-4-mini (Phi3ForCausalLM)
# Atenção: qkv_proj, o_proj | MLP: gate_up_proj, down_proj
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 4. Carregando dataset combinado (Anchor Mixing + GooseReason) formatado para ChatML Phi-4
dataset = load_dataset("json", data_files="cyberrott_final_train.jsonl", split="train")

def apply_template(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
            for msg in batch["messages"]
        ]
    }

dataset = dataset.map(apply_template, batched=True)

# 5. Configuração Dinâmica para compatibilidade total de versões do TRL (max_length vs max_seq_length)
sft_config_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,         # 2 para mitigar OOM com vocabulário de 200k
    "gradient_accumulation_steps": 8,         # Efetivo = batch_size * grad_accum = 16
    "gradient_checkpointing": True,           # Reduz VRAM de ativações de 38GB para ~14GB
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "report_to": "none",                      # Evita trava solicitando autenticação do wandb
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "bf16": torch.cuda.is_bf16_supported(),
    "optim": "adamw_torch_fused"
}

if "max_length" in sft_config_params:
    sft_kwargs["max_length"] = 4096
elif "max_seq_length" in sft_config_params:
    sft_kwargs["max_seq_length"] = 4096

if "dataset_text_field" in sft_config_params:
    sft_kwargs["dataset_text_field"] = "text"

training_args = SFTConfig(**sft_kwargs)

# Preparação do SFTTrainer (processing_class vs tokenizer)
trainer_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "peft_config": peft_config,
    "args": training_args,
}

sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in sft_kwargs:
    trainer_kwargs["dataset_text_field"] = "text"

trainer = SFTTrainer(**trainer_kwargs)

print("\nIniciando treinamento...")
trainer.train()

print("\nSalvando adaptadores...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Treinamento concluído e salvo em {OUTPUT_DIR}!")


In [ ]:
# 6. Teste de Inferência Pós-Treinamento (CyberRott em Ação)
from peft import PeftModel

print("Recarregando modelo com adaptadores para teste de inferência...")
test_model = PeftModel.from_pretrained(model, OUTPUT_DIR)

test_prompt = "Como funciona um ataque de Pass-the-Hash e quais as mitigações no Active Directory?"

chat = [
    {"role": "system", "content": "Você é o CyberRott, uma IA especialista em segurança ofensiva e defensiva com raciocínio aprofundado."}, 
    {"role": "user", "content": test_prompt}
]

prompt_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(test_model.device)

with torch.no_grad():
    out = test_model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.6,
        top_p=0.9,
        do_sample=True
    )

response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Resposta Gerada pelo CyberRott ---\n")
print(response)
